# Naive RAG from first principles

This notebook builds a complete baseline RAG system over eight fictional Northstar Labs policy documents. It uses only the Python standard library.

Pipeline: **question → one TF-IDF search → fixed top-k passages → prompt → answer**.

The notebook defaults to a deterministic offline generator. The final section shows how to switch only the generation step to the OpenAI Responses API.

## 1. Import the reusable implementation

In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "rag_core.py").exists():
    candidates = [ROOT / "naive-rag", ROOT.parent / "naive-rag"]
    ROOT = next(path for path in candidates if (path / "rag_core.py").exists())
sys.path.insert(0, str(ROOT))

from rag_core import NaiveRAG, TfidfIndex, build_prompt, chunk_documents, load_documents

print("Project root:", ROOT)

## 2. Load and inspect the synthetic documents

In [ ]:
documents = load_documents(ROOT / "documents")
print(f"Loaded {len(documents)} documents")
for document in documents:
    print(f"- {document.id}: {document.title}")

## 3. Split each policy into searchable chunks

In [ ]:
chunks = chunk_documents(documents)
print(f"Created {len(chunks)} chunks")
for chunk in chunks[:4]:
    print(f"\n{chunk.id} | {chunk.title}\n{chunk.text}")

## 4. Build the TF-IDF index and run one retrieval pass

In [ ]:
index = TfidfIndex(chunks)
question = "How many annual leave days can I carry over, and when do they expire?"
results = index.search(question, top_k=3)

for rank, result in enumerate(results, start=1):
    print(f"[{rank}] {result.chunk.title} | score={result.score:.4f}")
    print(result.chunk.text, "\n")

Notice the defining Naive RAG behavior: the original question is searched once, and the first three results are accepted without rewriting, reranking, or compression.

## 5. Construct the grounded prompt

In [ ]:
prompt = build_prompt(question, results)
print(prompt)

## 6. Generate an offline cited answer

In [ ]:
rag = NaiveRAG(ROOT / "documents", top_k=3)
response = rag.ask(question, generator="local")

print("ANSWER")
print(response["answer"])
print("\nTRACE")
print(json.dumps(response["trace"], indent=2))

## 7. Try several policy questions

In [ ]:
questions = [
    "How quickly must I report a suspected security incident?",
    "Do I need a receipt for a 250 rand taxi expense?",
    "How many weeks of leave does a secondary caregiver receive?",
]

for item in questions:
    result = rag.ask(item, generator="local")
    print("\nQ:", item)
    print("A:", result["answer"])
    print("Top source:", result["sources"][0]["title"])

## 8. Evaluate top-1 retrieval on a tiny labelled set

In [ ]:
evaluation = [
    ("annual leave carry over deadline", "annual-leave-policy"),
    ("report a suspected security incident", "information-security-policy"),
    ("receipt required for travel expense", "travel-expense-policy"),
    ("secondary caregiver paid leave", "parental-leave-policy"),
    ("return a company laptop after leaving", "equipment-policy"),
]

correct = 0
for query, expected_document in evaluation:
    top = rag.index.search(query, top_k=1)[0].chunk.document_id
    hit = top == expected_document
    correct += int(hit)
    print(f"{'PASS' if hit else 'FAIL'} | {query} → {top}")
print(f"Top-1 accuracy: {correct}/{len(evaluation)} = {correct/len(evaluation):.0%}")

## 9. Optional OpenAI generation

Set `OPENAI_API_KEY` in your environment, then set `RAG_GENERATOR=openai` or pass `generator="openai"`. Retrieval remains local; selected evidence is sent through the Responses API. Never paste a real key into the notebook.

In [ ]:
if os.environ.get("OPENAI_API_KEY"):
    openai_response = rag.ask(question, generator="openai")
    print(openai_response["answer"])
else:
    print("OPENAI_API_KEY is not set; the offline example is complete.")

## 10. Deploy the same pipeline

The adjacent `server.py` exposes the exact `NaiveRAG` object through a browser UI and JSON endpoints. Run `python server.py`, or build the provided Dockerfile.